# KNU 한국어 감성사전으로 상품평 감성분석하기

**환경:** Google Colab / Jupyter Notebook (인터넷 연결 필요)
**준비물:** `TS_상품평_4개카테고리_통합.xlsx` (상품평 21,404건)
**선수 학습:** 직접 만든 소형 사전으로 감성분석하기

## 오늘 할 일
지난 실습에서는 단어 35개짜리 **직접 만든 사전**을 썼습니다. 오늘은 대학 연구실이 만든 **1만 4천여 개 단어의 공개 감성사전**을 써 보고,
> "큰 사전을 쓰면 무조건 더 좋을까?"

를 직접 확인합니다.

## KNU 한국어 감성사전이란?
| 항목 | 내용 (공식 저장소 기준) |
|---|---|
| 만든 곳 | 군산대학교 소프트웨어융합공학과 Data Intelligence Lab |
| 크기 | 약 14,800개 항목 (단어, 여러 단어 표현, 축약어, 이모티콘) |
| 점수 | **−2(매우 부정) ~ +2(매우 긍정)** 다섯 단계 |
| 특징 | 특정 분야가 아닌 **일상적인 감정 표현** 중심 (공식 설명) |
| 저장소 | [github.com/park1200656/KnuSentiLex](https://github.com/park1200656/KnuSentiLex) |



## 실습 진행 과정
| 단계 | 할 일 | 핵심 메시지 |
|---|---|---|
| 준비 | 엑셀 읽기, 단어 나누기 | 부정어 `안`·`못`·`않다`를 남긴다 |
| ① 사전 불러오기 | 인터넷에서 사전 내려받기, 단어 검색 | 사전에 어떤 단어가 있는지 먼저 본다 |
| ② 감성 점수 매기기 | 단어 점수 합계 → 긍정/중립/부정 | 사전 점수를 그대로 더하면 된다 |
| ③ 결과 점검·사전 고치기 | 점수에 가장 큰 영향을 준 단어 확인 → ✏️ 잘못된 단어 고치기 | **범용 사전은 우리 데이터와 안 맞는 단어가 있다** |
| ④ 두 사전 비교 | 소형 사전 vs KNU 사전, 직접 채점 | 큰 사전이 항상 정답은 아니다 |
| 정리 | 결과 저장 | 사전은 데이터에 맞게 **고쳐 쓰는 것** |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.

---
## 0. 준비


In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 7.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 80.4 MB/s eta 0:00:00


Colab 왼쪽 **📁 파일 창**에 `TS_상품평_4개카테고리_통합.xlsx`를 끌어다 놓으세요.

In [2]:
# 표 다루기 도구 불러오기
import pandas as pd

# ===== ✏️ [입력] 엑셀 파일 이름 =====
excel_path = "TS_상품평_4개카테고리_통합.xlsx"
# ===================================

# 엑셀에서 카테고리와 상품평 두 열만 읽기
df = pd.read_excel(excel_path, usecols=["카테고리", "상품평"])
print("리뷰 수:", len(df))

리뷰 수: 21404


In [3]:
# Kiwi 형태소 분석기 불러오기
from kiwipiepy import Kiwi
kiwi = Kiwi()

# 남길 품사: 일반명사, 고유명사, 동사, 형용사, 어근
keep_tags = {"NNG", "NNP", "VV", "VA", "XR"}

# 형태소 분석 결과에서 필요한 단어만 골라 띄어쓰기로 이어 주는 함수
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag.split("-")[0]                  # 'VA-I' 같은 태그를 'VA'로
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")            # 동사·형용사·'않'은 '다' 붙이기
        elif tag == "XR":
            words.append(t.form + "하다")          # 어근은 '하다' 붙이기 (촉촉 → 촉촉하다)
        elif tag in keep_tags or t.form in {"안", "못"}:
            words.append(t.form)                  # 명사와 부정어 '안', '못'은 그대로
    return " ".join(words)

# 리뷰 전체를 한 번에 분석해서 '단어' 열 만들기 (20초 정도 걸림)
df["단어"] = [to_words(tokens) for tokens in kiwi.tokenize(df["상품평"].tolist())]
df[["상품평", "단어"]].head(3)

,상품평,단어
0,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사다 광고 말 향 은은하다 않다 올드 남자 스킨 로션 향 향
1,역시 전에 쓰던 게 좋은 것 같아요.\n향도 독하고 피부에 자극이 강해서 별로예요.,전 쓰다 좋다 같다 향 독하다 피부 자극 강하다
2,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요~~♡♡\n아참...,남편 선물 사다 저렴하다 향기 좋다 지인 추천 택배 상자 안 상자 찌그러지다 오다 ...


---
## ① 사전 불러오기
사전 파일은 한 줄에 `단어<탭>점수` 형태입니다. 인터넷 주소에서 바로 읽습니다.

In [4]:
# 표 형식 파일 읽기 옵션 도구 불러오기
import csv

# KNU 감성사전 파일 주소 (공식 GitHub 저장소)
url = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"

# 탭으로 구분된 파일을 표로 읽기 (따옴표를 특수 기호로 처리하지 않음)
lex_df = pd.read_csv(url, sep="\t", header=None, names=["단어", "점수"], quoting=csv.QUOTE_NONE)

# 점수가 비어 있는 줄 빼기, 단어 앞뒤 공백 정리, 중복 단어는 마지막 것만 남기기
lex_df = lex_df.dropna()
lex_df["단어"] = lex_df["단어"].str.strip()
lex_df = lex_df.drop_duplicates("단어", keep="last")

# {단어: 점수} 사전으로 바꾸기
knu = dict(zip(lex_df["단어"], lex_df["점수"].astype(int)))

# 사전 크기와 점수별 단어 수 출력
print("사전 크기:", len(knu))
lex_df["점수"].value_counts().sort_index()

사전 크기: 14851


,count
점수,
-2.0,4797
-1.0,5029
0.0,154
1.0,2268
2.0,2603


✅ 부정 단어(−1, −2)가 긍정 단어보다 **두 배쯤 많습니다.**

### 단어 검색해 보기
**✏️ [입력]** 궁금한 단어를 `search_words`에 넣어 보세요. 사전에 없으면 `없음`으로 나옵니다.

In [5]:
# ===== ✏️ [입력] 검색할 단어 =====
search_words = ["좋다", "만족", "별로", "아쉽다", "촉촉하다", "고장", "바르다", "크다"]
# ================================

# 단어마다 사전 점수 출력 (없으면 '없음')
for w in search_words:
    print(w, "→", knu.get(w, "없음"))

좋다 → 2
만족 → 2
별로 → 없음
아쉽다 → -1
촉촉하다 → 없음
고장 → 없음
바르다 → 2
크다 → 1


🤔 **이상한 점이 보이나요?**
- 상품평에 아주 많은 `별로`, `고장`, `촉촉하다`가 **사전에 없습니다.** 공식 설명대로 특정 분야보다 **일상 감정** 중심 사전이기 때문입니다.
- `바르다`는 **+2(매우 긍정)** 입니다. 사전은 "올바르다"의 뜻으로 보지만, 화장품 리뷰에서는 "로션을 **바르다**"라는 뜻입니다. 이런 단어를 **동음이의어**라고 합니다.

---
## ② 감성 점수 매기기 1

### 💡 규칙
1. 리뷰의 단어마다 사전 점수를 더합니다.
2. 사전에 단어가 없으면 **뒤에 `하다`를 붙여서** 한 번 더 찾습니다. (`만족` → `만족하다`)
3. 바로 앞에 `안`·`못`이 있거나 바로 뒤에 `않다`가 오면 **점수 부호를 뒤집습니다.** (`안 좋다` → −2)
4. 합계가 **0보다 크면 긍정, 0이면 중립, 0보다 작으면 부정**

> 💡 사전에서 `없다`는 −1이지만, `불편 없다`처럼 **앞 단어를 뒤집는 말**로 쓰이는 경우가 많아 점수에서 빼고 뒤집기 규칙에만 씁니다.

In [6]:
# 부호를 뒤집는 말
flip_before = {"안", "못"}       # 앞에 오면 뒤집기
flip_after = {"않다", "없다"}     # 뒤에 오면 뒤집기

# 리뷰 하나의 감성 점수를 계산하는 함수 (lexicon: 사용할 사전)
def sentiment_score(text, lexicon):
    words = text.split()
    total = 0
    for i, w in enumerate(words):
        # 뒤집는 말 자체는 점수에 넣지 않음
        if w in flip_before or w in flip_after:
            continue
        # 사전에서 점수 찾기 (없으면 '하다'를 붙여 한 번 더, 그래도 없으면 0)
        value = lexicon.get(w, lexicon.get(w + "하다", 0))
        if value == 0:
            continue
        # 앞 단어·뒤 단어 확인 후 부정어가 있으면 부호 뒤집기
        before = words[i - 1] if i > 0 else ""
        after = words[i + 1] if i + 1 < len(words) else ""
        if before in flip_before or after in flip_after:
            value = -value
        total += value
    return total

# 점수를 긍정/중립/부정으로 바꾸는 함수
def to_label(score):
    if score > 0:
        return "긍정"
    if score < 0:
        return "부정"
    return "중립"

# KNU 사전으로 모든 리뷰 점수 매기기
df["KNU점수"] = df["단어"].apply(lambda t: sentiment_score(t, knu))
df["KNU감성"] = df["KNU점수"].apply(to_label)

# 감성 비율(%) 출력
(df["KNU감성"].value_counts(normalize=True) * 100).round(1)

,proportion
KNU감성,
긍정,79.6
부정,13.9
중립,6.5


---
## ③ 결과 점검하고 사전 고치기

### 점수에 가장 큰 영향을 준 단어 찾기
"어떤 단어가 점수를 많이 올리고 내렸나?"를 보면 사전의 문제를 빨리 찾을 수 있습니다.
**영향력 = 단어 점수 × 등장 횟수**

In [7]:
# 단어 세기 도구 불러오기
from collections import Counter

# 모든 리뷰의 단어 등장 횟수 세기
counts = Counter(w for text in df["단어"] for w in text.split())

# 사전에 있는 단어만 골라 영향력 계산
rows = []
for w, n in counts.items():
    if w in flip_before or w in flip_after:
        continue
    value = knu.get(w, knu.get(w + "하다", 0))
    if value != 0:
        rows.append({"단어": w, "사전점수": value, "등장횟수": n, "영향력": value * n})
impact = pd.DataFrame(rows)

# 영향력이 큰 긍정 단어 15개, 부정 단어 15개를 나란히 보기
top_pos = impact.sort_values("영향력", ascending=False).head(15).reset_index(drop=True)
top_neg = impact.sort_values("영향력").head(15).reset_index(drop=True)
pd.concat([top_pos, top_neg], axis=1, keys=["점수를 올린 단어", "점수를 내린 단어"])

점수를 올린 단어                    점수를 내린 단어                 
          단어 사전점수   등장횟수    영향력        단어 사전점수  등장횟수   영향력
0         좋다    2  19387  38774       아쉽다   -1  2745 -2745
1         만족    2   6581  13162        불편   -2  1136 -2272
2        편하다    2   2066   4132       비싸다   -2   752 -1504
3       저렴하다    1   3013   3013       어렵다   -2   693 -1386
4         최고    2   1207   2414       힘들다   -2   449  -898
5       깔끔하다    2   1196   2392        실망   -2   443  -886
6         크다    1   2328   2328        걱정   -2   361  -722
7         편리    2   1123   2246      떨어지다   -1   576  -576
8         선명    1   2113   2113       약하다   -1   575  -575
9        바르다    2   1043   2086        이상   -1   536  -536
10      깨끗하다    2   1024   2048       느리다   -1   525  -525
11       빠르다    1   2035   2035        부담   -2   247  -494
12        많다    2    924   1848        불량   -2   206  -412
13        가능    2    761   1522       걸리다   -1   408  -408
14      간편하다    2    740   1480        짜증   -2   172  -344

🔍 **우리 데이터와 안 맞는 단어를 찾아보세요.** 예를 들면
| 단어 | 사전 점수 | 상품평에서의 실제 뜻 | 문제 |
|---|---|---|---|
| `바르다` | +2 | 로션을 **바르다** | 동음이의어 (사전은 "올바르다") |
| `크다` | +1 | 사이즈가 크다 (장점일 수도, 단점일 수도) | 분야마다 뜻이 다름 |
| `선` | +2 | 전원 **선**, 연결 **선** | 동음이의어 (사전은 "착할 선") |
| `이상` | −1 | 기대 **이상**, 1년 **이상** | 동음이의어 (사전은 "이상하다") |
| `받다` | +1 | 택배를 **받다** | 감정과 관계없는 말 |

### ✏️ 사전 고치기
KNU 사전을 **복사**한 뒤
- 우리 데이터와 맞지 않는 단어는 **0점으로** (= 무시)
- 사전에 없는데 상품평에 많은 감성 단어는 **추가**합니다.

In [8]:
# ===== ✏️ [입력] 사전 고치기 =====
# 0점으로 바꿀 단어 (우리 데이터에서 감정과 관계없는 뜻으로 쓰이는 단어)
ignore_words = ["바르다", "크다", "선", "이상", "받다", "많다", "가능", "높다", "걸리다", "먼지"]

# 추가할 단어와 점수 (−2 ~ +2)
add_words = {"별로": -2, "고장": -2, "촉촉하다": 1, "가성비": 1, "끈적이다": -1, "쉽다": 1, "불만": -2}
# ================================

# KNU 사전 복사본 만들기 (원본은 그대로 둠)
my_knu = dict(knu)

# 무시할 단어는 0점으로
for w in ignore_words:
    my_knu[w] = 0

# 새 단어 추가
my_knu.update(add_words)

# 고친 사전으로 다시 점수 매기기
df["고친점수"] = df["단어"].apply(lambda t: sentiment_score(t, my_knu))
df["고친감성"] = df["고친점수"].apply(to_label)

# 고치기 전후 감성 비율(%) 비교
pd.DataFrame({
    "KNU 원본": df["KNU감성"].value_counts(normalize=True),
    "고친 KNU": df["고친감성"].value_counts(normalize=True),
}).mul(100).round(1)

,KNU 원본,고친 KNU
긍정,79.6,78.0
부정,13.9,15.1
중립,6.5,6.9


In [9]:
# 고치기 전후로 감성이 바뀐 리뷰 수
changed = df[df["KNU감성"] != df["고친감성"]]
print("감성이 바뀐 리뷰 수:", len(changed))

# 바뀐 리뷰 예시 5개 (원문, 고치기 전, 고친 후)
changed[["상품평", "KNU감성", "고친감성"]].sample(5, random_state=1)

감성이 바뀐 리뷰 수: 1100


,상품평,KNU감성,고친감성
11053,네비에 이것 저것 터치해보니 사용하기 쉽고 편리하네요 화질이 떨어져 흐릿하게 보이니...,중립,긍정
19516,너무 커서 한쪽면만 사용하게 되어요. 그래도 많이 널 수 있습니다.,긍정,중립
7155,저음부는 우퍼의 둥둥 때려주는 음향이 듣기에 아주 좋습니다 :) \n전원 켜지는 소...,긍정,중립
12066,메모리 용량이 32기가인데 녹화가능시간이 7시간이네요ㅠ 화질도 생각했던것보다 선명하...,긍정,부정
16873,배송 빠르고 네비게이션 작동도 엄청 잘됩니다. 후방카메라까지 연동되니 너무 마음에 ...,긍정,중립


🤔 **고친 결과가 항상 더 맞을까?** 바뀐 리뷰를 하나씩 읽어 보세요.
- 어떤 리뷰는 엉뚱한 단어(`바르다`, `크다`)가 빠져서 **더 정확**해집니다.
- 반대로 "많이 널 수 있어요"처럼 `많다`가 진짜 장점이었던 리뷰는 **중립으로 바뀌어** 오히려 덜 정확해질 수 있습니다.

사전을 고칠 때는 **고친 결과도 다시 확인**해야 합니다.

---
## ④ 두 사전 비교하기
지난 실습의 **직접 만든 소형 사전**과 비교합니다. 소형 사전은 점수가 모두 ±1이라 같은 형태로 바꿔서 같은 함수로 계산합니다.

In [10]:
# 지난 실습의 소형 사전 (긍정 +1, 부정 -1)
small_pos = ["좋다", "만족", "최고", "저렴하다", "편하다", "빠르다", "깔끔하다", "튼튼하다", "선명",
             "추천", "괜찮다", "촉촉하다", "은은하다", "편리", "가성비", "훌륭하다", "간편하다", "쉽다"]
small_neg = ["별로", "아쉽다", "불편", "비싸다", "느리다", "불만", "실망", "고장", "최악",
             "나쁘다", "귀찮다", "어렵다", "답답하다", "약하다", "끈적이다", "소음", "불량"]
small = {w: 1 for w in small_pos} | {w: -1 for w in small_neg}

# 소형 사전으로 점수 매기기
df["소형감성"] = df["단어"].apply(lambda t: to_label(sentiment_score(t, small)))

# 세 가지 사전의 감성 비율(%) 비교
pd.DataFrame({
    "소형 사전(35개)": df["소형감성"].value_counts(normalize=True),
    "KNU 원본": df["KNU감성"].value_counts(normalize=True),
    "고친 KNU": df["고친감성"].value_counts(normalize=True),
}).mul(100).round(1)

,소형 사전(35개),KNU 원본,고친 KNU
긍정,74.5,79.6,78.0
부정,11.3,13.9,15.1
중립,14.3,6.5,6.9


In [11]:
# 두 방법의 결과가 같은 비율(%)
print("소형 vs KNU 원본 일치율:", round((df["소형감성"] == df["KNU감성"]).mean() * 100, 1), "%")
print("소형 vs 고친 KNU 일치율:", round((df["소형감성"] == df["고친감성"]).mean() * 100, 1), "%")

소형 vs KNU 원본 일치율: 82.8 %
소형 vs 고친 KNU 일치율: 84.7 %


### ✍️ 직접 채점하기
어느 사전이 더 맞는지는 **사람이 확인**해야 알 수 있습니다.
1. 아래 셀로 리뷰 10개를 읽습니다.
2. 내 판단을 `my_labels`에 순서대로 적습니다. (`"긍정"`, `"중립"`, `"부정"`)
3. 다음 셀을 실행하면 사전별로 몇 개 맞혔는지 나옵니다.

In [12]:
# 무작위 리뷰 10개 뽑기 (매번 같은 10개)
check = df.sample(10, random_state=7)

# 번호와 리뷰 출력
for n, text in enumerate(check["상품평"], 1):
    print(n, text[:100])

1 초기 등록 할 때 지문인식이 조금 번거러웠으나 불편한 정도는 아니였어요. 고속도로 달릴 때 소리가 조금 작은감이 있어서 최대 볼륨을 더 키울수 있으면 좋겠네요.
2 어마무시하게 스크린이 크네요 완전 박진감 넘치는 화면을 즐길 수 있습니다
다만 화면 한쪽 끝이 초점이 잘 안맞아서 불편하네요 ㅠㅠ
그래도 4K의 최고급화질이 정말 좋습니다^^
대신
3 무게가 매우 가벼워 이동이 너무 쉽고 좋아요.가성비 최고입니다
4 여자 혼자서도 금방 조립하네요 제 옷이 큰데 넓게 널수 있어서 넘 좋습니다
5 남편 말로는 괜찮은 거 같다고 합니다. 자세히 얘기하진 않는데 잘 스며들고 피부가 떙기지 않아서 좋다고 합니다. 그전에 비싸게 샀던 거랑 비교해도 품질이 별 차이 안 난다고 하네요
6 고민했지만 품질이 우수해서 정말 잘산듯해요 ㅎㅎ 그리고 화면도 진짜커요 만족합니다^^
7 좋은 제품 저렴하게 잘 구매했구요~ 설치도 깔끔하게 잘 해 주셔서 너무 감사합니다~아쉬운건 디자인이 생각보다 촌스러워서 좀 당황했네요
8 가격저렴하고 완제품배송이라 맘에 들어요 근데 지지대가 프라스틱이라 튼튼할지 의문이 들긴하네요 전체적으로 만족해요
9 화질은 선명해서 깨끗하게 잘 보이는게 굉장히 좋아요^^  하지만 라이브 녹화는 버튼 누르면 10초 전후로 설정 되어 있는게 너무 짧아서 길었으면 하는 아쉬움이 있어요ㅠㅠ
10 설치가 생각보다 간단하여 좋습니다^^
화질도 타사대비 선명도가 훨씬 높습니다. 강추!


In [13]:
# ===== ✏️ [입력] 위 10개 리뷰에 대한 내 판단 (순서대로 10개) =====
my_labels = []
my_labels = ["긍정", "긍정", "부정", "중립", "긍정", "긍정", "부정", "긍정", "긍정", "중립"]
# ================================================================

# 10개를 모두 적었으면 사전별 맞힌 개수 계산
if len(my_labels) == 10:
    check["내 판단"] = my_labels
    for col in ["소형감성", "KNU감성", "고친감성"]:
        print(f"{col}: {(check[col] == check['내 판단']).sum()} / 10")
    display(check[["상품평", "내 판단", "소형감성", "KNU감성", "고친감성"]])
else:
    print("my_labels에 판단 10개를 적은 뒤 다시 실행하세요.")

소형감성: 4 / 10
KNU감성: 5 / 10
고친감성: 5 / 10


,상품평,내 판단,소형감성,KNU감성,고친감성
15638,초기 등록 할 때 지문인식이 조금 번거러웠으나 불편한 정도는 아니였어요. 고속도로 ...,긍정,중립,중립,중립
7503,어마무시하게 스크린이 크네요 완전 박진감 넘치는 화면을 즐길 수 있습니다\n다만 화...,긍정,중립,긍정,긍정
3114,무게가 매우 가벼워 이동이 너무 쉽고 좋아요.가성비 최고입니다,부정,긍정,긍정,긍정
19369,여자 혼자서도 금방 조립하네요 제 옷이 큰데 넓게 널수 있어서 넘 좋습니다,중립,긍정,긍정,긍정
1071,남편 말로는 괜찮은 거 같다고 합니다. 자세히 얘기하진 않는데 잘 스며들고 피부가 ...,긍정,긍정,긍정,긍정
4858,고민했지만 품질이 우수해서 정말 잘산듯해요 ㅎㅎ 그리고 화면도 진짜커요 만족합니다^^,긍정,긍정,긍정,긍정
12273,좋은 제품 저렴하게 잘 구매했구요~ 설치도 깔끔하게 잘 해 주셔서 너무 감사합니다~...,부정,긍정,긍정,긍정
19813,가격저렴하고 완제품배송이라 맘에 들어요 근데 지지대가 프라스틱이라 튼튼할지 의문이 ...,긍정,긍정,긍정,긍정
12474,화질은 선명해서 깨끗하게 잘 보이는게 굉장히 좋아요^^ 하지만 라이브 녹화는 버튼...,긍정,긍정,긍정,긍정
16102,설치가 생각보다 간단하여 좋습니다^^\n화질도 타사대비 선명도가 훨씬 높습니다. 강추!,중립,긍정,긍정,긍정


---
## 정리

In [14]:
# 세 사전의 결과를 CSV 파일로 저장 (한글 깨짐 방지 인코딩)
df[["카테고리", "상품평", "소형감성", "KNU감성", "고친감성"]].to_csv("knu_sentiment_result.csv", index=False, encoding="utf-8-sig")
print("저장 완료: knu_sentiment_result.csv")

저장 완료: knu_sentiment_result.csv


### ✅ 오늘의 한 줄 정리
**공개 감성사전은 좋은 출발점이지만, 우리 데이터에 맞게 고쳐 써야 한다.**

| 비교 | 소형 사전 (직접 만든 것) | KNU 원본 | 고친 KNU |
|---|---|---|---|
| 크기 | 35개 | 약 14,800개 | 약 14,800개 + 수정 |
| 장점 | 우리 데이터에 딱 맞음 | 많은 단어, 강도(±1, ±2) 구분 | 둘의 장점 |
| 단점 | 놓치는 단어가 많음 | 분야가 달라 엉뚱한 단어가 점수에 들어감 | 고치는 데 사람의 시간이 듦 |

### 🤔 생각해 볼 점
- **여러 단어 표현을 못 씀:** KNU 사전의 절반 이상은 `가치 있다`, `가격이 싸다` 같은 **여러 단어 표현**입니다. 오늘 방법은 단어 하나씩만 찾아서 이 표현들은 대부분 쓰이지 않습니다. 원문 문장에서 직접 찾는 방법도 있지만, 조사·어미 모양이 조금만 달라도 맞지 않는 문제가 있습니다.
- **동음이의어:** `바르다`, `선`, `이상`처럼 같은 글자에 뜻이 여러 개인 단어는 사전만으로 구별할 수 없습니다. 문맥을 이해하는 방법(단원 5 임베딩, 단원 6 LLM)이 필요합니다.
- **누구의 판단이 정답인가:** 직접 채점 결과도 사람마다 다를 수 있습니다. 실무에서는 여러 사람이 채점하고 의견이 다를 때 토론해서 정답을 정합니다. KNU 사전도 최소 3명의 평가자가 판단하는 방식으로 만들어졌다고 공식 문서에 적혀 있습니다.



**참고**
- [KNU 한국어 감성사전 (GitHub)](https://github.com/park1200656/KnuSentiLex)